In [1]:
import pandas as pd
import numpy as np

### Data Loading

In [2]:
df = pd.read_csv("data/geo-reviews-dataset-2023.csv")
df

,address,name_ru,rating,rubrics,text
0,"Екатеринбург, ул. Московская / ул. Волгоградск...",Московский квартал,3.0,Жилой комплекс,Московский квартал 2.\nШумно : летом по ночам ...
1,"Московская область, Электросталь, проспект Лен...",Продукты Ермолино,5.0,Магазин продуктов;Продукты глубокой заморозки;...,"Замечательная сеть магазинов в общем, хороший ..."
2,"Краснодар, Прикубанский внутригородской округ,...",LimeFit,1.0,Фитнес-клуб,"Не знаю смутят ли кого-то данные правила, но я..."
3,"Санкт-Петербург, проспект Энгельса, 111, корп. 1",Snow-Express,4.0,Пункт проката;Прокат велосипедов;Сапсёрфинг,Хорошие условия аренды. \nДружелюбный персонал...
4,"Тверь, Волоколамский проспект, 39",Студия Beauty Brow,5.0,"Салон красоты;Визажисты, стилисты;Салон бровей...",Топ мастер Ангелина топ во всех смыслах ) Немн...
...,...,...,...,...,...
499995,"Москва, Южный административный округ, район Би...",Бирюлёво-Пассажирская,4.0,Железнодорожная станция,"Охрана кривая но добрая, двери не закрываются ..."
499996,"Москва, Южный административный округ, район Би...",Бирюлёво-Пассажирская,4.0,Железнодорожная станция,По сравнению со многими современными платформа...
499997,"Новосибирск, Коммунистическая улица, 48А",NaN,5.0,"Бар, паб","Приятная атмосфера, прекрасное вино, волшебная..."
499998,"Астраханская область, Харабалинский район",Сарай-Бату,5.0,Достопримечательность,Был с семьёй 13.06.23 Отличное место. Рекоменд...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   address  500000 non-null  object 
 1   name_ru  499030 non-null  object 
 2   rating   500000 non-null  float64
 3   rubrics  500000 non-null  object 
 4   text     500000 non-null  object 
dtypes: float64(1), object(4)
memory usage: 19.1+ MB


In [4]:
pd.concat([df.rating.value_counts(), round(df.rating.value_counts()/len(df)* 100, 2)], axis=1)

,count,count
rating,,
5.0,390515,78.10
4.0,41160,8.23
1.0,34351,6.87
3.0,21686,4.34
2.0,12088,2.42
0.0,200,0.04


In [5]:
df_prepared = df[(df.rating <=2) | (df.rating >=4)].copy()

print(df_prepared.shape)
df_prepared = df_prepared.drop_duplicates(subset=['text'], keep='first').copy()
print(df_prepared.shape)

df_prepared['label'] = (df_prepared.rating <= 2 ).astype(int)

df_prepared = df_prepared.reset_index(drop=True)

(478314, 5)
(478199, 5)


In [6]:
df_prepared

,address,name_ru,rating,rubrics,text,label
0,"Московская область, Электросталь, проспект Лен...",Продукты Ермолино,5.0,Магазин продуктов;Продукты глубокой заморозки;...,"Замечательная сеть магазинов в общем, хороший ...",0
1,"Краснодар, Прикубанский внутригородской округ,...",LimeFit,1.0,Фитнес-клуб,"Не знаю смутят ли кого-то данные правила, но я...",1
2,"Санкт-Петербург, проспект Энгельса, 111, корп. 1",Snow-Express,4.0,Пункт проката;Прокат велосипедов;Сапсёрфинг,Хорошие условия аренды. \nДружелюбный персонал...,0
3,"Тверь, Волоколамский проспект, 39",Студия Beauty Brow,5.0,"Салон красоты;Визажисты, стилисты;Салон бровей...",Топ мастер Ангелина топ во всех смыслах ) Немн...,0
4,"Иркутская область, Черемхово, Первомайская ули...",Tele2,5.0,Оператор сотовой связи;Интернет-провайдер,"Приятное общение, все доступно объяснили, мне ...",0
...,...,...,...,...,...,...
478194,"Москва, Южный административный округ, район Би...",Бирюлёво-Пассажирская,4.0,Железнодорожная станция,"Охрана кривая но добрая, двери не закрываются ...",0
478195,"Москва, Южный административный округ, район Би...",Бирюлёво-Пассажирская,4.0,Железнодорожная станция,По сравнению со многими современными платформа...,0
478196,"Новосибирск, Коммунистическая улица, 48А",NaN,5.0,"Бар, паб","Приятная атмосфера, прекрасное вино, волшебная...",0
478197,"Астраханская область, Харабалинский район",Сарай-Бату,5.0,Достопримечательность,Был с семьёй 13.06.23 Отличное место. Рекоменд...,0


In [7]:
pd.concat([
    df_prepared.label.value_counts(), 
    round(df_prepared.label.value_counts(normalize=True), 3)*100
    ],
    axis=1
)

,count,proportion
label,,
0,431560,90.2
1,46639,9.8


### Data splitting

In [8]:
from sklearn.model_selection import train_test_split

df_train_val, df_test = train_test_split(
    df_prepared,
    test_size=.2,
    random_state=42,
    stratify=df_prepared.label
)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=.125,
    random_state=42,
    stratify=df_train_val.label
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [9]:
df_train

,address,name_ru,rating,rubrics,text,label
0,"Москва, улица Поляны, 22",Бык да Баран,4.0,"Ресторан;Пекарня;Бар, паб;Кафе;Банкетный зал","Вкусно, разнообразно , немного дороговато\n",0
1,"Москва, Болотная набережная, 15",ГЭС-2,5.0,Дом культуры;Достопримечательность,"невероятное пространство - кинематографичное, ...",0
2,"Санкт-Петербург, Артиллерийская улица, 1Р",Пешкарус,5.0,Экскурсии;Квесты,Познакомиться с историческами местами Казани н...,0
3,"Калининград, ул. Тихоненко",Тихий Дом,5.0,Жилой комплекс,Со мной работал менеджер Александр. Он все оче...,0
4,"Республика Башкортостан, Стерлитамак, улица Ху...",Elcigara,5.0,Вейп-шоп,большое спасибо эльсигар!!! очень дёшево и кач...,0
...,...,...,...,...,...,...
334734,"Вологда, Пошехонское шоссе, 22",Колизеум,5.0,Компьютерный клуб;Интернет-кафе;Игровой клуб;К...,Место просто кайф! Очень атмосферное и приятно...,0
334735,"Санкт-Петербург, Пушкин, Железнодорожная улица...",Детскосельская,4.0,Железнодорожный вокзал,"Не очень удобные платформы, узкие. Грубо говор...",0
334736,"Московская область, Химки, Железнодорожная ули...",Каравай-СВ,5.0,Пекарня,"Хлебо-булочные изделия у Каравая-СВ вкусные, а...",0
334737,"Ростов-на-Дону, улица Герасименко, 5",Altego,5.0,Салон красоты;Спа-салон,Побывала здесь сегодня на спа процедуре «крист...,0


In [10]:
print(f"Total sample size: {df_prepared.shape[0]} reviews")
print(f"\nSamples sizes:")
print(f"    Train:       {df_train.shape[0]} ({df_train.shape[0]/df_prepared.shape[0]*100:.2f}%)")
print(f"    Validation:  {df_val.shape[0]}  ({df_val.shape[0]/df_prepared.shape[0]*100:.2f}%)")
print(f"    Test:        {df_test.shape[0]}  ({df_test.shape[0]/df_prepared.shape[0]*100:.2f}%)")

print(f"\nClass(label) proportion:")
print(df_train.label.value_counts(normalize=True).sort_index())

Total sample size: 478199 reviews

Samples sizes:
    Train:       334739 (70.00%)
    Validation:  47820  (10.00%)
    Test:        95640  (20.00%)

Class(label) proportion:
label
0    0.90247
1    0.09753
Name: proportion, dtype: float64


### Immports

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
from tqdm.auto import tqdm
import os
import random
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import seaborn as sns
import matplotlib.pyplot as plt

### Model hyperparameters

In [12]:
MODEL_NAME = "intfloat/multilingual-e5-small"
BATCH_SIZE = 256
MAX_LENGTH = 64
NUM_EPOCHS = 10
RANDOM_SEED = 42

### Seeds set-up

In [13]:
def seed_all(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(RANDOM_SEED)

### Tokienizer

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [15]:
input_texts = [
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.",
    "There is no one who loves pain itself, who seeks after it and wants to have it, simply because it is pain.",
    "Нет никого, кто любил бы саму боль, кто искал бы её и желал бы её иметь, просто потому, что это боль.",
    "没有谁会因为痛苦本身而热爱痛苦、追求痛苦并想要拥有痛苦"
]
encoded_texts = []

for text in input_texts:
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )
    encoded_texts.append(encoding)

In [16]:
for key, val in encoded_texts[0].items():
    print(f"{key} : {val}")
    print(val.shape, '\n')

input_ids : tensor([[    0,  7069,  3601,  3052,  1661,  2760,     4,  6783,  8340,  3870,
             4,  1736,    54, 56618, 15042, 64451,   486, 27554,    82, 17196,
          9521, 34122,     5,     2,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1]])
torch.Size([1, 64]) 

attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
torch.Size([1, 64]) 



In [30]:
sample = encoded_texts[1]
tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'][0])
attention_mask = sample['attention_mask'][0]
input_ids = sample['input_ids'][0].tolist()

real_tokens = [t for t, m in zip(tokens, attention_mask) if m == 1]
real_ids = [id_ for id_, m in zip(input_ids, attention_mask) if m == 1]

pd.DataFrame([real_tokens], columns=real_ids)

,0,8622,83,110,1632,2750,5161,7,24503,68034,...,765,442,4,42856,6637,442,83,24503,5,2
0,<s>,▁There,▁is,▁no,▁one,▁who,▁love,s,▁pain,▁itself,...,▁have,▁it,",",▁simply,▁because,▁it,▁is,▁pain,.,</s>


### TextDataset class

In [31]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64):
        super().__init__()
        self.texts = df.text.tolist()
        self.labels = df.label.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.float)
        }

#### DataSets initialization 

In [32]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextDataset(
    df_train,
    tokenizer
)

val_dataset = TextDataset(
    df_val,
    tokenizer
)

test_dataset = TextDataset(
    df_test,
    tokenizer
)

In [38]:
train_dataset[1337]

{'input_ids': tensor([     0,   2718,  24276,   2233,  35343,   7405,    135, 190948,  43525,
            419,     35,    440,   1343,   6467,  21013,    827,     29,  31346,
              6, 117243,    227,      2,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'label': tensor(0.)}

In [39]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [50]:
batch = next(iter(train_loader))
batch, batch['input_ids'].shape, batch['attention_mask'].shape, batch['label'].shape, len(train_loader)

({'input_ids': tensor([[     0,   3858,  83510,  ...,      1,      1,      1],
          [     0,  98278,   2042,  ...,      1,      1,      1],
          [     0,   1087,    197,  ...,      1,      1,      1],
          ...,
          [     0,   2429,   2033,  ...,      1,      1,      1],
          [     0,  55155,   5119,  ...,  24840,   7843,      2],
          [     0, 195154,  19776,  ...,      1,      1,      1]]),
  'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          ...,
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 1, 1, 1],
          [1, 1, 1,  ..., 0, 0, 0]]),
  'label': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.

### Selecting a model for fine-tuning